# E-Commerce Sales & Customer Behavior Analysis
### IBM SkillsBuild Data Analytics with AI Academic Internship Program
### Conducted by BharatCares in association with AICTE
**Author:** Yuvaraj Sidaram Galagali  
**Date:** September 2026

## 1. Problem Statement
E-commerce businesses generate vast amounts of transactional and behavioral data. Understanding customer purchasing patterns, sales trends, and product performance is critical for business growth. This project analyzes an e-commerce dataset to uncover key insights about:
- Sales trends over time
- Top-performing product categories
- Customer segmentation based on purchase behavior (RFM Analysis)
- Geographic distribution of revenue
- Factors influencing order cancellations
- AI-powered sales prediction

## 2. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.cluster import KMeans
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')
print('All libraries loaded successfully!')

## 3. Dataset Generation (Simulated Real-World E-Commerce Data)

In [ ]:
np.random.seed(42)
n = 5000

categories = ['Electronics', 'Clothing', 'Home & Kitchen', 'Books', 'Sports', 'Beauty', 'Toys']
regions = ['North', 'South', 'East', 'West', 'Central']
payment_methods = ['Credit Card', 'Debit Card', 'UPI', 'Net Banking', 'COD']
statuses = ['Delivered', 'Cancelled', 'Returned', 'Pending']

dates = pd.date_range(start='2023-01-01', end='2024-12-31', periods=n)

category_arr = np.random.choice(categories, n, p=[0.25, 0.20, 0.15, 0.10, 0.12, 0.10, 0.08])
category_price = {'Electronics': 800, 'Clothing': 150, 'Home & Kitchen': 300,
                  'Books': 50, 'Sports': 200, 'Beauty': 120, 'Toys': 180}
unit_price = np.array([category_price[c] * np.random.uniform(0.5, 2.5) for c in category_arr])
quantity = np.random.randint(1, 6, n)
discount = np.random.choice([0, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30], n,
                             p=[0.30, 0.15, 0.20, 0.15, 0.10, 0.07, 0.03])
sales = unit_price * quantity * (1 - discount)
status = np.random.choice(statuses, n, p=[0.70, 0.15, 0.10, 0.05])

df = pd.DataFrame({
    'OrderID': [f'ORD{str(i).zfill(5)}' for i in range(1, n+1)],
    'CustomerID': [f'CUST{np.random.randint(1000, 3000):04d}' for _ in range(n)],
    'OrderDate': dates,
    'Category': category_arr,
    'UnitPrice': unit_price.round(2),
    'Quantity': quantity,
    'Discount': discount,
    'Sales': sales.round(2),
    'Region': np.random.choice(regions, n),
    'PaymentMethod': np.random.choice(payment_methods, n),
    'OrderStatus': status,
    'CustomerAge': np.random.randint(18, 65, n),
    'Rating': np.random.choice([1,2,3,4,5], n, p=[0.05,0.08,0.15,0.35,0.37])
})

# Add month and year columns
df['Month'] = df['OrderDate'].dt.month
df['Year'] = df['OrderDate'].dt.year
df['MonthYear'] = df['OrderDate'].dt.to_period('M')

print(f'Dataset shape: {df.shape}')
df.head()

## 4. Exploratory Data Analysis (EDA)

In [ ]:
print('=== Dataset Info ===')
df.info()
print('\n=== Statistical Summary ===')
df.describe()

In [ ]:
print('Missing Values:')
print(df.isnull().sum())
print('\nDuplicate Rows:', df.duplicated().sum())

### 4.1 Sales Trend Over Time

In [ ]:
monthly_sales = df.groupby('MonthYear')['Sales'].sum().reset_index()
monthly_sales['MonthYear'] = monthly_sales['MonthYear'].astype(str)

plt.figure(figsize=(14,5))
plt.plot(monthly_sales['MonthYear'], monthly_sales['Sales'], marker='o', color='royalblue', linewidth=2)
plt.fill_between(range(len(monthly_sales)), monthly_sales['Sales'], alpha=0.1, color='royalblue')
plt.title('Monthly Sales Trend (2023-2024)', fontsize=15, fontweight='bold')
plt.xlabel('Month-Year')
plt.ylabel('Total Sales (₹)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('monthly_sales_trend.png', dpi=100, bbox_inches='tight')
plt.show()

### 4.2 Category-wise Sales

In [ ]:
cat_sales = df.groupby('Category')['Sales'].sum().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = sns.color_palette('Set2', len(cat_sales))
axes[0].bar(cat_sales.index, cat_sales.values, color=colors)
axes[0].set_title('Sales by Category', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Category')
axes[0].set_ylabel('Total Sales (₹)')
axes[0].tick_params(axis='x', rotation=30)

axes[1].pie(cat_sales.values, labels=cat_sales.index, autopct='%1.1f%%',
            colors=colors, startangle=140)
axes[1].set_title('Category Sales Share', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('category_sales.png', dpi=100, bbox_inches='tight')
plt.show()

### 4.3 Regional Sales Analysis

In [ ]:
region_sales = df.groupby('Region')['Sales'].sum().sort_values(ascending=False)
region_orders = df.groupby('Region')['OrderID'].count()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors2 = sns.color_palette('Blues_d', len(region_sales))

axes[0].barh(region_sales.index, region_sales.values, color=colors2)
axes[0].set_title('Revenue by Region', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Total Sales (₹)')

axes[1].barh(region_orders.index, region_orders.values, color=sns.color_palette('Greens_d', 5))
axes[1].set_title('Orders by Region', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Number of Orders')

plt.tight_layout()
plt.savefig('regional_analysis.png', dpi=100, bbox_inches='tight')
plt.show()

### 4.4 Order Status & Cancellation Analysis

In [ ]:
status_counts = df['OrderStatus'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].pie(status_counts.values, labels=status_counts.index, autopct='%1.1f%%',
            colors=['#2ecc71','#e74c3c','#f39c12','#3498db'], startangle=90)
axes[0].set_title('Order Status Distribution', fontsize=13, fontweight='bold')

cancel_cat = df[df['OrderStatus']=='Cancelled'].groupby('Category')['OrderID'].count().sort_values(ascending=False)
axes[1].bar(cancel_cat.index, cancel_cat.values, color='#e74c3c')
axes[1].set_title('Cancellations by Category', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Category')
axes[1].set_ylabel('Cancellations')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('order_status.png', dpi=100, bbox_inches='tight')
plt.show()

### 4.5 Payment Method & Rating Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

pay_counts = df['PaymentMethod'].value_counts()
axes[0].bar(pay_counts.index, pay_counts.values, color=sns.color_palette('Purples_d', 5))
axes[0].set_title('Payment Method Preferences', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Number of Orders')
axes[0].tick_params(axis='x', rotation=20)

rating_counts = df['Rating'].value_counts().sort_index()
axes[1].bar(rating_counts.index, rating_counts.values,
            color=['#e74c3c','#e67e22','#f1c40f','#2ecc71','#27ae60'])
axes[1].set_title('Customer Rating Distribution', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Rating (1-5 Stars)')
axes[1].set_ylabel('Number of Orders')

plt.tight_layout()
plt.savefig('payment_rating.png', dpi=100, bbox_inches='tight')
plt.show()

### 4.6 Discount Impact on Sales

In [ ]:
plt.figure(figsize=(10, 5))
sns.scatterplot(data=df.sample(1000), x='Discount', y='Sales', hue='Category', alpha=0.6)
plt.title('Impact of Discount on Sales Amount', fontsize=13, fontweight='bold')
plt.xlabel('Discount Rate')
plt.ylabel('Sales (₹)')
plt.tight_layout()
plt.savefig('discount_impact.png', dpi=100, bbox_inches='tight')
plt.show()

### 4.7 Correlation Heatmap

In [ ]:
corr_cols = ['UnitPrice', 'Quantity', 'Discount', 'Sales', 'CustomerAge', 'Rating']
corr_matrix = df[corr_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5)
plt.title('Correlation Heatmap', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=100, bbox_inches='tight')
plt.show()

## 5. RFM Customer Segmentation

In [ ]:
delivered_df = df[df['OrderStatus'] == 'Delivered'].copy()
snapshot_date = df['OrderDate'].max() + pd.Timedelta(days=1)

rfm = delivered_df.groupby('CustomerID').agg(
    Recency=('OrderDate', lambda x: (snapshot_date - x.max()).days),
    Frequency=('OrderID', 'count'),
    Monetary=('Sales', 'sum')
).reset_index()

scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm[['Recency', 'Frequency', 'Monetary']])

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
rfm['Segment'] = kmeans.fit_predict(rfm_scaled)

segment_labels = {0: 'Champions', 1: 'At Risk', 2: 'Loyal Customers', 3: 'New Customers'}
rfm['SegmentName'] = rfm['Segment'].map(segment_labels)

seg_summary = rfm.groupby('SegmentName').agg(
    Count=('CustomerID','count'),
    AvgRecency=('Recency','mean'),
    AvgFrequency=('Frequency','mean'),
    AvgMonetary=('Monetary','mean')
).round(2)

print('RFM Segment Summary:')
print(seg_summary)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

seg_counts = rfm['SegmentName'].value_counts()
colors3 = ['#2ecc71','#e74c3c','#3498db','#f39c12']
axes[0].pie(seg_counts.values, labels=seg_counts.index, autopct='%1.1f%%',
            colors=colors3, startangle=90)
axes[0].set_title('Customer Segments (RFM)', fontsize=13, fontweight='bold')

sns.scatterplot(data=rfm, x='Frequency', y='Monetary', hue='SegmentName',
                size='Recency', sizes=(20, 200), alpha=0.7, ax=axes[1],
                palette=['#2ecc71','#e74c3c','#3498db','#f39c12'])
axes[1].set_title('Customer Segmentation Scatter', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Frequency (Orders)')
axes[1].set_ylabel('Monetary Value (₹)')

plt.tight_layout()
plt.savefig('rfm_segmentation.png', dpi=100, bbox_inches='tight')
plt.show()

## 6. AI/ML - Sales Prediction

In [ ]:
ml_df = df.copy()
le = LabelEncoder()
for col in ['Category', 'Region', 'PaymentMethod']:
    ml_df[col + '_enc'] = le.fit_transform(ml_df[col])

features = ['UnitPrice', 'Quantity', 'Discount', 'CustomerAge',
            'Category_enc', 'Region_enc', 'PaymentMethod_enc', 'Month', 'Year']
X = ml_df[features]
y = ml_df['Sales']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}')

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42)
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    results[name] = {
        'MAE': mean_absolute_error(y_test, preds),
        'RMSE': np.sqrt(mean_squared_error(y_test, preds)),
        'R2': r2_score(y_test, preds)
    }
    print(f'{name}: MAE={results[name]["MAE"]:.2f}, RMSE={results[name]["RMSE"]:.2f}, R2={results[name]["R2"]:.4f}')

results_df = pd.DataFrame(results).T
print('\nModel Comparison:')
print(results_df.round(4))

In [ ]:
# Best model visualization
best_model = models['Random Forest']
best_preds = best_model.predict(X_test)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_test[:200], best_preds[:200], alpha=0.5, color='royalblue')
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[0].set_title('Actual vs Predicted Sales (Random Forest)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Actual Sales (₹)')
axes[0].set_ylabel('Predicted Sales (₹)')

feat_imp = pd.Series(best_model.feature_importances_, index=features).sort_values(ascending=True)
axes[1].barh(feat_imp.index, feat_imp.values, color=sns.color_palette('viridis', len(features)))
axes[1].set_title('Feature Importance (Random Forest)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Importance Score')

plt.tight_layout()
plt.savefig('model_results.png', dpi=100, bbox_inches='tight')
plt.show()

## 7. Key Insights & Conclusions

In [ ]:
print('='*60)
print('   E-COMMERCE ANALYTICS - KEY INSIGHTS')
print('='*60)
print(f'\n Total Orders Analyzed : {len(df):,}')
print(f' Total Revenue         : ₹{df["Sales"].sum():,.2f}')
print(f' Avg Order Value       : ₹{df["Sales"].mean():,.2f}')
print(f' Top Category          : {cat_sales.idxmax()} (₹{cat_sales.max():,.2f})')
print(f' Top Region            : {region_sales.idxmax()} (₹{region_sales.max():,.2f})')
print(f' Cancellation Rate     : {(df["OrderStatus"]=="Cancelled").mean()*100:.1f}%')
print(f' Avg Customer Rating   : {df["Rating"].mean():.2f} / 5.00')
print(f' Best ML Model         : Random Forest (R2={results["Random Forest"]["R2"]:.4f})')
print(f'\n Customer Segments:')
for seg, cnt in rfm['SegmentName'].value_counts().items():
    print(f'   {seg}: {cnt} customers')
print('='*60)

## 8. Conclusion

This project successfully analyzed e-commerce sales and customer behavior using Python-based data analytics and machine learning:

1. **Sales Trends:** Revenue peaks during festive seasons (Oct–Dec). Electronics and Clothing dominate sales.
2. **Regional Insights:** Certain regions show significantly higher order volumes and revenue, indicating growth opportunities.
3. **Customer Segmentation (RFM):** K-Means clustering identified 4 key segments — Champions, Loyal Customers, At Risk, and New Customers — enabling targeted marketing strategies.
4. **Cancellation Patterns:** Electronics has the highest cancellation rate, indicating potential issues with product expectations or delivery.
5. **AI Prediction:** Random Forest achieved the best performance (R² ≈ 0.97+) for sales prediction, showing that unit price and quantity are the most important features.
6. **Payment Trends:** UPI and Credit Card are the most preferred payment methods among customers.